# Getting the Bronze Data From External storage

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
import builtins
from pyspark.sql.functions import current_date, lit
import builtins

base_path = "abfss://orderpayments@adlsstdout001tgt.dfs.core.windows.net/Payments/"
# ---- Year ----
folders = dbutils.fs.ls(base_path)
latest_year = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Month ----
folders = dbutils.fs.ls(f"{base_path}{latest_year}/")
latest_month = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Day ----
folders = dbutils.fs.ls(f"{base_path}{latest_year}/{latest_month}/")
latest_day = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Final Path ----
latest_path = f"{base_path}{latest_year}/{latest_month}/{latest_day}/"

print(f"Reading from: {latest_path}")

# ---- File Date ----
file_date = f"{latest_year}-{latest_month}-{latest_day}"

# ---- Read parquet ----
raw_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(base_path)
    .withColumn("ingest_date", current_date())
    .withColumn("file_date", lit(file_date))
)

display(raw_df.limit(10))

# Data Cleaning

In [0]:
import builtins
from pyspark.sql.functions import current_date, lit
import builtins
from pyspark.sql.functions import current_date, lit
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ==========================================
# STEP 1 — STANDARDIZE COLUMN NAMES
# ==========================================

clean_df = raw_df.toDF(
    *[col.lower().strip().replace(" ", "_") for col in raw_df.columns]
)

# ==========================================
# STEP 2 — REMOVE DUPLICATES
# ==========================================

clean_df = clean_df.dropDuplicates()

# ==========================================
# STEP 3 — TRIM STRING COLUMNS
# ==========================================

string_cols = ["order_id", "payment_type"]

for c in string_cols:
    clean_df = clean_df.withColumn(c, trim(col(c)))

# ==========================================
# STEP 4 — REMOVE SPECIAL CHARACTERS
# (only if required by business)
# ==========================================

clean_df = (
    clean_df
    .withColumn(
        "order_id",
        regexp_replace(col("order_id"), "[^A-Za-z0-9]", "")
    )
)

# ==========================================
# STEP 5 — STANDARDIZE PAYMENT TYPE
# ==========================================

clean_df = (
    clean_df
    .withColumn("payment_type", lower(col("payment_type")))
)

# ==========================================
# STEP 6 — HANDLE NULLS
# ==========================================

clean_df = (
    clean_df
    .filter(col("order_id").isNotNull())
    .filter(col("payment_type").isNotNull())
)

# ==========================================
# STEP 7 — DATA TYPE CASTING
# ==========================================

clean_df = (
    clean_df
    .withColumn(
        "payment_sequential",
        col("payment_sequential").cast(IntegerType())
    )
    .withColumn(
        "payment_installments",
        col("payment_installments").cast(IntegerType())
    )
    .withColumn(
        "payment_value",
        round(col("payment_value").cast(DecimalType(10,2)), 2)
    )
    .withColumn(
        "ingest_date",
        to_date(col("ingest_date"))
    )
    .withColumn(
        "file_date",
        to_date(col("file_date"))
    )
)

# ==========================================
# STEP 8 — BUSINESS RULE VALIDATIONS
# ==========================================

clean_df = (
    clean_df
    .filter(col("payment_value") > 0)
    .filter(col("payment_installments") > 0)
    .filter(col("payment_sequential") > 0)
)

# ==========================================
# STEP 9 — VALID PAYMENT TYPES
# ==========================================

valid_payment_types = [
    "credit_card",
    "boleto",
    "voucher",
    "debit_card"
]

clean_df = clean_df.filter(
    col("payment_type").isin(valid_payment_types)
)

# ==========================================
# STEP 10 — ADD AUDIT COLUMNS
# ==========================================

clean_df = (
    clean_df
    .withColumn("created_timestamp", current_timestamp())
    .withColumn("updated_timestamp", current_timestamp())
)

# ==========================================
# STEP 11 — DATA QUALITY FLAG
# ==========================================

clean_df = (
    clean_df
    .withColumn(
        "data_quality_flag",
        when(col("payment_value") <= 0, "REJECTED")
        .otherwise("VALID")
    )
)

# ==========================================
# STEP 12 — FINAL COLUMN ORDER
# ==========================================

clean_df = clean_df.select(
    "order_id",
    "payment_sequential",
    "payment_type",
    "payment_installments",
    "payment_value",
    "ingest_date",
    "file_date",
    "created_timestamp",
    "updated_timestamp",
    "data_quality_flag"
)

# ==========================================
# SHOW RESULTS
# ==========================================

display(clean_df)

In [0]:
tablename = "ecom.silver.fact_silver_payments"

In [0]:
(
        clean_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option(
            "path",
            "abfss://silver@silverprocessdbstorage.dfs.core.windows.net/Payments"
        )
        .saveAsTable(tablename)
    )

print("Full Load Completed Successfully")